In [ ]:
import pybaseball as pyb
import pandas as pd
import numpy as np

# ------------------- CONFIGURATION -------------------
years = range(2021, 2026)                  # 2021–2025
pitch_type_filter = 'SL'                   # slider
stand_filter = 'L'                         # LHB
p_throws_filter = 'L'                      # LHP

# Spin rate bins
spin_bins = [0, 1800, 2100, 2400, 2700, 3000, 3300, 9999]
spin_labels = ['<1800', '1800-2100', '2100-2400', '2400-2700', 
               '2700-3000', '3000-3300', '3300+']

results = []

for year in years:
    print(f"Processing year {year}...")
    
    try:
        data_year = pyb.statcast(start_dt=f'{year}-03-01', end_dt=f'{year}-11-01')
    except Exception as e:
        print(f"Error for {year}: {e}")
        continue
    
    if data_year.empty:
        continue
    
    # Early filtering to save memory
    filt = (
        (data_year['pitch_type'] == pitch_type_filter) &
        (data_year['stand'] == stand_filter) &
        (data_year['p_throws'] == p_throws_filter) &
        (data_year['zone'].between(1, 9)) &                  # in-zone only
        (data_year['events'].notna()) &                      # batted balls
        (data_year['estimated_woba_using_speedangle'].notna())
    )
    
    df_filt = data_year[filt][[
        'zone', 
        'release_spin_rate', 
        'estimated_woba_using_speedangle',
        'launch_speed', 
        'launch_angle'
    ]].copy()
    
    if df_filt.empty:
        print(f"No data for {year}")
        continue
    
    # Bin spin rate
    df_filt['spin_bin'] = pd.cut(
        df_filt['release_spin_rate'],
        bins=spin_bins,
        labels=spin_labels,
        include_lowest=True,
        right=False
    )
    
    # Aggregation
    agg = df_filt.groupby(['zone', 'spin_bin']).agg(
        count=('zone', 'size'),
        avg_xwoba=('estimated_woba_using_speedangle', 'mean'),
        avg_ev=('launch_speed', 'mean'),
        avg_la=('launch_angle', 'mean')
    ).round(3).reset_index()
    
    agg['year_range'] = f"{years[0]}-{years[-1]}"
    results.append(agg)
    
    del data_year, df_filt  # free memory

# Combine all years
if results:
    final = pd.concat(results)
    final_agg = final.groupby(['zone', 'spin_bin']).agg(
        total_count=('count', 'sum'),
        avg_xwoba=('avg_xwoba', 'mean'),
        avg_ev=('avg_ev', 'mean'),
        avg_la=('avg_la', 'mean')
    ).round(3).reset_index()
    
    print("\n=== Average xwOBA for sliders LHB vs LHP (2021–2025) ===")
    print(final_agg.sort_values(['zone', 'spin_bin']))
    
    # Save to CSV
    final_agg.to_csv('slider_xwoba_by_zone_spin_LvL_2021-2025.csv', index=False)
    print("Saved to CSV")
else:
    print("No data after filtering")